<a href="https://colab.research.google.com/github/dudl1/-/blob/main/MODEL_FOR_COMMERSE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import re

In [92]:
# ==================== ТОКЕНИЗАЦИЯ И ПОДГОТОВКА ДАННЫХ ====================

class Tokenizer:
    """Токенизатор для преобразования текста в числовые последовательности"""

    def __init__(self, min_freq=1):
        self.min_freq = min_freq
        self.word2idx = {}
        self.idx2word = {}
        self.vocab_size = 0

        # Специальные токены
        self.PAD_TOKEN = "<PAD>"
        self.UNK_TOKEN = "<UNK>"
        self.BOS_TOKEN = "<BOS>"
        self.EOS_TOKEN = "<EOS>"

    def fit(self, text):
        """Создание словаря на основе текста"""
        # Простая токенизация
        words = self._tokenize(text)

        # Подсчёт частоты слов
        word_counts = Counter(words)

        # Создание словаря
        self.word2idx = {
            self.PAD_TOKEN: 0,
            self.UNK_TOKEN: 1,
            self.BOS_TOKEN: 2,
            self.EOS_TOKEN: 3
        }

        idx = 4
        for word, count in word_counts.items():
            if count >= self.min_freq:
                self.word2idx[word] = idx
                idx += 1

        self.idx2word = {idx: word for word, idx in self.word2idx.items()}
        self.vocab_size = len(self.word2idx)

        print(f"Размер словаря: {self.vocab_size}")
        return self

    def _tokenize(self, text):
        """Разбиение текста на токены"""
        # Приведение к нижнему регистру и разбиение на слова
        text = text.lower()
        # Разделение на слова и знаки препинания
        tokens = re.findall(r'\b\w+\b|[.,!?;:]', text)
        return tokens

    def encode(self, text, add_special_tokens=True):
        """Преобразование текста в индексы"""
        tokens = self._tokenize(text)

        if add_special_tokens:
            tokens = [self.BOS_TOKEN] + tokens + [self.EOS_TOKEN]

        indices = [self.word2idx.get(token, self.word2idx[self.UNK_TOKEN])
                   for token in tokens]
        return indices

    def decode(self, indices, skip_special_tokens=True):
        """Преобразование индексов обратно в текст"""
        special_tokens = {self.PAD_TOKEN, self.UNK_TOKEN,
                          self.BOS_TOKEN, self.EOS_TOKEN}

        words = []
        for idx in indices:
            word = self.idx2word.get(idx, self.UNK_TOKEN)
            if skip_special_tokens and word in special_tokens:
                continue
            words.append(word)

        return ' '.join(words)


class TextDataset(Dataset):
    """Датасет для языковой модели"""

    def __init__(self, text, tokenizer, seq_length=50):
        self.tokenizer = tokenizer
        self.seq_length = seq_length

        # Кодируем весь текст
        self.encoded = tokenizer.encode(text, add_special_tokens=False)

        # Создаём пары (входная последовательность, целевая последовательность)
        self.samples = []
        for i in range(0, len(self.encoded) - seq_length - 1, seq_length // 2):
            input_seq = self.encoded[i:i + seq_length]
            target_seq = self.encoded[i + 1:i + seq_length + 1]
            if len(input_seq) == seq_length and len(target_seq) == seq_length:
                self.samples.append((input_seq, target_seq))

        print(f"Создано {len(self.samples)} обучающих примеров")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        input_seq, target_seq = self.samples[idx]
        return (torch.tensor(input_seq, dtype=torch.long),
                torch.tensor(target_seq, dtype=torch.long))


# ==================== МЕХАНИЗМ ВНИМАНИЯ ====================

class Attention(nn.Module):
    """
    Механизм внимания (Bahdanau/Additive Attention)
    Позволяет модели фокусироваться на важных частях последовательности
    """

    def __init__(self, hidden_size, attention_size=None):
        super(Attention, self).__init__()

        if attention_size is None:
            attention_size = hidden_size

        self.hidden_size = hidden_size
        self.attention_size = attention_size

        # Линейные преобразования для запроса и ключей
        self.query_layer = nn.Linear(hidden_size, attention_size)
        self.key_layer = nn.Linear(hidden_size, attention_size)

        # Вектор для вычисления скоров внимания
        self.energy_layer = nn.Linear(attention_size, 1)

    def forward(self, query, keys, mask=None):
        """
        Args:
            query: текущее скрытое состояние [batch, hidden_size]
            keys: все скрытые состояния [batch, seq_len, hidden_size]
            mask: маска для паддинга [batch, seq_len]

        Returns:
            context: взвешенная сумма [batch, hidden_size]
            attention_weights: веса внимания [batch, seq_len]
        """
        batch_size, seq_len, _ = keys.size()

        # Преобразуем запрос: [batch, hidden] -> [batch, 1, attention_size]
        query_transformed = self.query_layer(query).unsqueeze(1)

        # Преобразуем ключи: [batch, seq_len, attention_size]
        keys_transformed = self.key_layer(keys)

        # Вычисляем энергию: [batch, seq_len, attention_size]
        energy = torch.tanh(query_transformed + keys_transformed)

        # Скоры внимания: [batch, seq_len]
        attention_scores = self.energy_layer(energy).squeeze(-1)

        # Применяем маску (если есть)
        if mask is not None:
            attention_scores = attention_scores.masked_fill(mask == 0, -1e9)

        # Нормализуем через softmax
        attention_weights = F.softmax(attention_scores, dim=-1)

        # Вычисляем контекстный вектор: [batch, hidden_size]
        context = torch.bmm(attention_weights.unsqueeze(1), keys).squeeze(1)

        return context, attention_weights


class MultiHeadSelfAttention(nn.Module):
    """
    Многоголовое самовнимание (Multi-Head Self-Attention)
    Позволяет модели обращать внимание на разные аспекты последовательности
    """

    def __init__(self, hidden_size, num_heads=4, dropout=0.1):
        super(MultiHeadSelfAttention, self).__init__()

        assert hidden_size % num_heads == 0, "hidden_size должен делиться на num_heads"

        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.head_size = hidden_size // num_heads

        # Проекции для Q, K, V
        self.query = nn.Linear(hidden_size, hidden_size)
        self.key = nn.Linear(hidden_size, hidden_size)
        self.value = nn.Linear(hidden_size, hidden_size)

        # Выходная проекция
        self.output = nn.Linear(hidden_size, hidden_size)

        self.dropout = nn.Dropout(dropout)
        self.scale = np.sqrt(self.head_size)

    def forward(self, x, mask=None):
        """
        Args:
            x: входная последовательность [batch, seq_len, hidden_size]
            mask: маска для каузального внимания [batch, 1, seq_len, seq_len]
        """
        batch_size, seq_len, _ = x.size()

        # Вычисляем Q, K, V
        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)

        # Разделяем на головы: [batch, num_heads, seq_len, head_size]
        Q = Q.view(batch_size, seq_len, self.num_heads, self.head_size).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.num_heads, self.head_size).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_heads, self.head_size).transpose(1, 2)

        # Скоры внимания: [batch, num_heads, seq_len, seq_len]
        scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale

        # Применяем каузальную маску (чтобы не видеть будущее)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)

        # Веса внимания
        attention_weights = F.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)

        # Применяем внимание к значениям
        context = torch.matmul(attention_weights, V)

        # Объединяем головы: [batch, seq_len, hidden_size]
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, -1)

        # Выходная проекция
        output = self.output(context)

        return output, attention_weights


# ==================== ЯЗЫКОВАЯ МОДЕЛЬ ====================

class LSTMLanguageModelWithAttention(nn.Module):
    """
    Языковая модель на основе LSTM с механизмом внимания
    """

    def __init__(self, vocab_size, embedding_size=256, hidden_size=512,
                 num_layers=2, num_heads=4, dropout=0.3, use_self_attention=True):
        super(LSTMLanguageModelWithAttention, self).__init__()

        self.vocab_size = vocab_size
        self.embedding_size = embedding_size
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.use_self_attention = use_self_attention

        # Слой эмбеддингов
        self.embedding = nn.Embedding(vocab_size, embedding_size, padding_idx=0)

        # LSTM слои
        self.lstm = nn.LSTM(
            input_size=embedding_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=False
        )

        # Механизмы внимания
        if use_self_attention:
            self.self_attention = MultiHeadSelfAttention(
                hidden_size=hidden_size,
                num_heads=num_heads,
                dropout=dropout
            )

        self.attention = Attention(hidden_size)

        # Слой нормализации
        self.layer_norm = nn.LayerNorm(hidden_size)

        # Выходной слой
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size * 2, vocab_size)  # *2 из-за конкатенации с контекстом

        # Инициализация весов
        self._init_weights()

    def _init_weights(self):
        """Инициализация весов модели"""
        for name, param in self.named_parameters():
            if 'weight' in name and 'embedding' not in name:
                if len(param.shape) >= 2:
                    nn.init.xavier_uniform_(param)
            elif 'bias' in name:
                nn.init.zeros_(param)

        # Специальная инициализация для эмбеддингов
        nn.init.uniform_(self.embedding.weight, -0.1, 0.1)

    def create_causal_mask(self, seq_len, device):
        """Создание каузальной маски для самовнимания"""
        mask = torch.tril(torch.ones(seq_len, seq_len, device=device))
        return mask.unsqueeze(0).unsqueeze(0)  # [1, 1, seq_len, seq_len]

    def forward(self, x, hidden=None, return_attention=False):
        """
        Args:
            x: входные индексы токенов [batch, seq_len]
            hidden: начальное скрытое состояние LSTM
            return_attention: возвращать ли веса внимания
        """
        batch_size, seq_len = x.size()
        device = x.device

        # Эмбеддинги: [batch, seq_len, embedding_size]
        embedded = self.embedding(x)
        embedded = self.dropout(embedded)

        # LSTM: [batch, seq_len, hidden_size]
        lstm_output, hidden = self.lstm(embedded, hidden)

        # Самовнимание (опционально)
        if self.use_self_attention:
            # Создаём каузальную маску
            causal_mask = self.create_causal_mask(seq_len, device)

            # Применяем самовнимание
            attention_output, self_attention_weights = self.self_attention(
                lstm_output, mask=causal_mask
            )

            # Residual connection + Layer Normalization
            lstm_output = self.layer_norm(lstm_output + attention_output)
        else:
            self_attention_weights = None

        # Внимание к предыдущим состояниям для каждой позиции
        outputs = []
        attention_weights_list = []

        for t in range(seq_len):
            # Текущее скрытое состояние: [batch, hidden_size]
            current_hidden = lstm_output[:, t, :]

            # Контекст из предыдущих состояний
            if t > 0:
                # Используем все предыдущие состояния как ключи
                keys = lstm_output[:, :t, :]
                context, attn_weights = self.attention(current_hidden, keys)
                attention_weights_list.append(attn_weights)
            else:
                # Для первого токена нет предыдущих состояний
                context = torch.zeros_like(current_hidden)

            # Объединяем текущее состояние с контекстом
            combined = torch.cat([current_hidden, context], dim=-1)
            outputs.append(combined)

        # Собираем выходы: [batch, seq_len, hidden_size * 2]
        outputs = torch.stack(outputs, dim=1)
        outputs = self.dropout(outputs)

        # Прогнозы: [batch, seq_len, vocab_size]
        logits = self.fc(outputs)

        if return_attention:
            return logits, hidden, attention_weights_list, self_attention_weights

        return logits, hidden

    def generate(self, tokenizer, prompt="", max_length=100, temperature=1.0,
                 top_k=50, top_p=0.9, device='cpu'):
        """
        Генерация текста

        Args:
            tokenizer: токенизатор
            prompt: начальный текст
            max_length: максимальная длина генерации
            temperature: температура сэмплирования (выше = более случайный)
            top_k: количество топ токенов для сэмплирования
            top_p: nucleus sampling порог
            device: устройство для вычислений
        """
        self.eval()

        # Кодируем промпт
        if prompt:
            input_ids = tokenizer.encode(prompt, add_special_tokens=False)
        else:
            input_ids = [tokenizer.word2idx[tokenizer.BOS_TOKEN]]

        generated = input_ids.copy()
        hidden = None

        with torch.no_grad():
            for _ in range(max_length):
                # Подготавливаем вход
                x = torch.tensor([generated[-50:]], dtype=torch.long, device=device)

                # Получаем прогнозы
                logits, hidden = self(x, hidden)

                # Берём прогноз для последнего токена
                next_token_logits = logits[0, -1, :] / temperature

                # Top-k фильтрация
                if top_k > 0:
                    indices_to_remove = next_token_logits < torch.topk(next_token_logits, top_k)[0][-1]
                    next_token_logits[indices_to_remove] = -float('inf')

                # Top-p (nucleus) фильтрация
                if top_p < 1.0:
                    sorted_logits, sorted_indices = torch.sort(next_token_logits, descending=True)
                    cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)

                    sorted_indices_to_remove = cumulative_probs > top_p
                    sorted_indices_to_remove[1:] = sorted_indices_to_remove[:-1].clone()
                    sorted_indices_to_remove[0] = 0

                    indices_to_remove = sorted_indices[sorted_indices_to_remove]
                    next_token_logits[indices_to_remove] = -float('inf')

                # Сэмплируем следующий токен
                probs = F.softmax(next_token_logits, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1).item()

                generated.append(next_token)

                # Останавливаемся на EOS токене
                if next_token == tokenizer.word2idx[tokenizer.EOS_TOKEN]:
                    break

        return tokenizer.decode(generated, skip_special_tokens=True)


# ==================== ОБУЧЕНИЕ ====================

class Trainer:
    """Класс для обучения модели"""

    def __init__(self, model, tokenizer, device='cpu'):
        self.model = model.to(device)
        self.tokenizer = tokenizer
        self.device = device

        # Оптимизатор с различными learning rate для разных частей модели
        self.optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=0.001,
            weight_decay=0.01
        )

        # Планировщик learning rate
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, mode='min', factor=0.5, patience=2
        )

        # Функция потерь (игнорируем PAD токен)
        self.criterion = nn.CrossEntropyLoss(ignore_index=0)

    def train_epoch(self, dataloader, clip_grad=1.0):
        """Одна эпоха обучения"""
        self.model.train()
        total_loss = 0
        num_batches = 0

        for batch_idx, (input_seq, target_seq) in enumerate(dataloader):
            input_seq = input_seq.to(self.device)
            target_seq = target_seq.to(self.device)

            # Прямой проход
            self.optimizer.zero_grad()
            logits, _ = self.model(input_seq)

            # Вычисляем потери
            # logits: [batch, seq_len, vocab_size]
            # target: [batch, seq_len]
            loss = self.criterion(
                logits.view(-1, logits.size(-1)),
                target_seq.view(-1)
            )

            # Обратный проход
            loss.backward()

            # Gradient clipping для предотвращения взрыва градиентов
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), clip_grad)

            # Обновление весов
            self.optimizer.step()

            total_loss += loss.item()
            num_batches += 1

            if batch_idx % 10 == 0:
                print(f"  Batch {batch_idx}, Loss: {loss.item():.4f}")

        avg_loss = total_loss / num_batches
        perplexity = np.exp(avg_loss)

        return avg_loss, perplexity

    def train(self, train_data, epochs=10, batch_size=32, seq_length=50):
        """Полный цикл обучения"""

        # Создаём датасет и загрузчик
        dataset = TextDataset(train_data, self.tokenizer, seq_length=seq_length)
        dataloader = DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=True,
            num_workers=0
        )

        print(f"\nНачало обучения на {self.device}")
        print(f"Эпох: {epochs}, Batch size: {batch_size}, Seq length: {seq_length}")
        print("-" * 50)

        best_loss = float('inf')

        for epoch in range(epochs):
            print(f"\nЭпоха {epoch + 1}/{epochs}")

            # Обучение
            train_loss, train_perplexity = self.train_epoch(dataloader)

            print(f"  Средний Loss: {train_loss:.4f}")
            print(f"  Perplexity: {train_perplexity:.2f}")

            # Обновляем learning rate
            self.scheduler.step(train_loss)

            # Сохраняем лучшую модель
            if train_loss < best_loss:
                best_loss = train_loss
                torch.save(self.model.state_dict(), 'best_model.pt')
                print("  Модель сохранена!")

            # Генерируем пример текста
            if (epoch + 1) % 2 == 0:
                print("\n  Пример генерации:")
                sample = self.model.generate(
                    self.tokenizer,
                    prompt="",
                    max_length=30,
                    temperature=0.8,
                    device=self.device
                )
                print(f"  -> {sample}")

        print("\nОбучение завершено!")
        return best_loss


# ==================== ОСНОВНОЙ КОД ====================

def main(training_text):
    """
    Основная функция для создания и обучения модели

    Args:
        training_text: текст для обучения модели
    """
    # Определяем устройство
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Используемое устройство: {device}")

    # Создаём токенизатор
    print("\n1. Создание токенизатора...")
    tokenizer = Tokenizer(min_freq=1)
    tokenizer.fit(training_text)

    # Создаём модель
    print("\n2. Создание модели...")
    model = LSTMLanguageModelWithAttention(
        vocab_size=tokenizer.vocab_size,
        embedding_size=200,
        hidden_size=256,
        num_layers=4,
        num_heads=32,
        dropout=0.2,
        use_self_attention=True
    )

    # Подсчёт параметров
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Всего параметров: {total_params:,}")
    print(f"Обучаемых параметров: {trainable_params:,}")

    # Создаём тренер и обучаем
    print("\n3. Обучение модели...")
    trainer = Trainer(model, tokenizer, device=device)
    trainer.train(
        training_text,
        epochs=45,
        batch_size=12,
        seq_length=150
    )

    # Генерация без промпта
    print("\nГенерация без промпта:")
    for i in range(1):
        generated = model.generate(
            tokenizer,
            prompt="",
            max_length=50,
            temperature=0.7,
            top_k=40,
            top_p=0.9,
            device=device
        )
        print(f"{i+1}. {generated}")

    return model, tokenizer, trainer

In [93]:
if __name__ == "__main__":
    # Пример обучающих данных (замените на свои)
    TRAINING_TEXT = '''

    '''

    # Запуск обучения
    model, tokenizer, trainer = main(TRAINING_TEXT)

Используемое устройство: cuda

1. Создание токенизатора...
Размер словаря: 17718

2. Создание модели...
Всего параметров: 15,076,455
Обучаемых параметров: 15,076,455

3. Обучение модели...
Создано 1456 обучающих примеров

Начало обучения на cuda
Эпох: 45, Batch size: 12, Seq length: 150
--------------------------------------------------

Эпоха 1/45
  Batch 0, Loss: 9.7825
  Batch 10, Loss: 7.3688
  Batch 20, Loss: 7.3001
  Batch 30, Loss: 7.1558
  Batch 40, Loss: 7.2162
  Batch 50, Loss: 7.3483
  Batch 60, Loss: 7.3383
  Batch 70, Loss: 7.1880
  Batch 80, Loss: 7.3306
  Batch 90, Loss: 7.3714
  Batch 100, Loss: 7.1678
  Batch 110, Loss: 7.2238
  Batch 120, Loss: 7.2438
  Средний Loss: 7.3559
  Perplexity: 1565.42
  Модель сохранена!

Эпоха 2/45
  Batch 0, Loss: 7.1040
  Batch 10, Loss: 7.0268
  Batch 20, Loss: 7.0596
  Batch 30, Loss: 7.1837
  Batch 40, Loss: 6.8772
  Batch 50, Loss: 7.3663
  Batch 60, Loss: 7.0527
  Batch 70, Loss: 7.0103
  Batch 80, Loss: 6.9348
  Batch 90, Loss: 7.1

In [96]:
prompts = ["Бедняга свидетель выронил чашку, бутерброд и сам упал"]

for prompt in prompts:
  generated = model.generate(
    tokenizer,
    prompt=prompt,
    max_length=100,
    temperature=0.7,
    device=trainer.device
  )
print(f"Результат: {generated}")

Результат: бедняга свидетель выронил чашку , бутерброд и сам упал другу на пойдем , получает за каждой более жить назад на говорил , где лошади страшно дороги от чудесный погоду , которые лицо могут могут мастера тихо . мне быть , кого обезьяна таскать таскать шагов всем капли . так если только хотите ! повторил он наконец мастера мастера присущей как какой какой спокойны ! мастер виноградинка так высочеству снова так только только в замке , предложил мастер виноградинка . мастер виноградинка так не довольно затылок . кум тыква : нужно только вовремя крышу , шиворот и головой вынужден без помощи ; только держи дух , а поспешно почувствовал в
